# 3. RQ3: Reuse without a package manager

**Methodology step 3: quantitative analysis.**

* RQ3.1 How concentrated is reuse? Gini coefficient and share of occurrences held by the top 1% of skills.
* RQ3.2 Does copying cross owners (ecosystem reuse) or stay within one owner?
* RQ3.3 How many edited variants do popular skills have? Families are grouped by normalised skill name (baseline); `reuse.minhash_families` groups by content similarity.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
print("database:", os.environ["GITSKILLS_DB"])

In [ ]:
skills = clean.load_skills()
strict = clean.strict(skills)

## 3.1 Concentration

In [ ]:
conc = pd.DataFrame([{"population": "all", **reuse.concentration(skills["copies"])},
                     {"population": "strict", **reuse.concentration(strict["copies"])}])
save_table(conc, "rq3_concentration")
conc.T

## 3.2 Cross-owner reuse

In [ ]:
dup = strict[strict["copies"] > 1]
print(f"copied skills: {len(dup):,}; spread across >1 owner: {(dup.n_owners > 1).mean():.1%}")
strict.nlargest(15, "copies")[["name", "copies", "n_repos", "n_owners", "location_class", "has_scripts"]]

## 3.3 Variant families

In [ ]:
fam = reuse.name_families(strict)
sizes = fam.groupby("family").agg(variants=("file_sha", "nunique"), occurrences=("copies", "sum"))
print(f"{len(sizes):,} families with >=2 variants, covering {sizes.variants.sum():,} skills")
sizes.sort_values("variants", ascending=False).head(15)